In [1]:
!pip install -q "bitsandbytes>=0.46.1" peft accelerate sentence-transformers rank_bm25 rouge_score statsmodels datasets
!pip install -q "numpy>=1.24,<2.3" "pandas>=2.0,<2.3"

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 55.8 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
from datasets import load_dataset

t = load_dataset("Tobi-Bueck/customer-support-tickets", split="train").to_pandas()

t = t[t["language"].str.lower().str.startswith("en")]
t = t[["body", "answer"]].dropna()
t = t[(t["body"].str.len() > 30) & (t["answer"].str.len() > 30)]
t = t.drop_duplicates().sample(frac=1, random_state=42).reset_index(drop=True)

test_queries = t["body"][:100].tolist()
reference_answers = t["answer"][:100].tolist()
enterprise_docs = t["answer"][100:2100].tolist()

README.md:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

aa_dataset-tickets-multi-lang-5-2-50-ver(…): reconstructing file:   0%|          |  0.00B / 26.0MB            

aa_dataset-tickets-multi-lang-5-2-50-ver(…): downloading bytes:           |  0.00B            

(…)set-tickets-german_normalized_50_5_2.csv:   0%|          | 0.00/8.33M [00:00<?, ?B/s]

dataset-tickets-multi-lang-4-20k.csv: reconstructing file:   0%|          |  0.00B / 18.8MB            

dataset-tickets-multi-lang-4-20k.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/61765 [00:00<?, ? examples/s]

In [5]:
from typing import List, Dict, Any
import numpy as np
import torch
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, util, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Cross-encoder reranker (used by retrieve_all in the evaluation cells)
print("Loading Cross-Encoder Reranker...")
reranker_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')


class QwenGenerator:
    def __init__(self, model_id: str = "Qwen/Qwen2.5-1.5B-Instruct", adapter_path: str = None):
        print("Loading Qwen generator (4-bit)...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16,
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_id, quantization_config=bnb_config, device_map={"": 0},
            torch_dtype=torch.float16,
        )
        if adapter_path:
            from peft import PeftModel
            model = PeftModel.from_pretrained(model, adapter_path)
        self.model = model.eval()

    @torch.inference_mode()
    def generate(self, query: str, context_chunks: List[str], multi_hop: bool = False,
                 max_new_tokens: int = 150) -> str:
        if context_chunks:
            context = "\n".join(f"[{i+1}] {c[:700]}" for i, c in enumerate(context_chunks))
            system = ("You are a precise enterprise IT support assistant. Answer the user's ticket using the "
                      "provided past support replies. Do not copy them word for word; write a clear, helpful reply.")
            if multi_hop:
                system += " Combine the relevant information from all the replies into one coherent answer."
            user = f"Past support replies:\n{context}\n\nTicket: {query[:1500]}"
        else:
            system = ("You are a precise enterprise IT support assistant. No relevant past support replies "
                      "are available. Answer briefly from general IT knowledge and say so if you are unsure.")
            user = f"Ticket: {query[:1500]}"

        messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        out = self.model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                  repetition_penalty=1.1, pad_token_id=self.tokenizer.eos_token_id)
        return self.tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()


class TechnicalDenseStore:
    def __init__(self, documents: List[str]):
        self.documents = documents
        self.encoder = SentenceTransformer('all-MiniLM-L6-v2')
        self.doc_embeddings = self.encoder.encode(documents, convert_to_tensor=True,
                                                  batch_size=64, show_progress_bar=False)

    def search_scored(self, query: str, top_k: int = 5):
        query_embedding = self.encoder.encode(query, convert_to_tensor=True)
        scores = util.cos_sim(query_embedding, self.doc_embeddings)[0].cpu().numpy()
        idx = np.argsort(scores)[::-1][:top_k]
        return [self.documents[i] for i in idx], [float(scores[i]) for i in idx], idx


class TechnicalSparseStore:
    def __init__(self, documents: List[str]):
        self.documents = documents
        tokenized_corpus = [doc.lower().split() for doc in documents]
        self.bm25 = BM25Okapi(tokenized_corpus)

    def search_scored(self, query: str, top_k: int = 5):
        scores = np.array(self.bm25.get_scores(query.lower().split()))
        idx = np.argsort(scores)[::-1][:top_k]
        return [self.documents[i] for i in idx], [float(scores[i]) for i in idx], idx


class EnterpriseAdaptiveRAGPipeline:
    """Holds the corpus stores and retrieval helpers.
    End-to-end answering is done by the evaluation cells
    (retrieve_all plus the generator), not by a run() method.
    """

    def __init__(self, technical_corpus: List[str], generator: QwenGenerator = None):
        self.documents = technical_corpus
        self.dense_store = TechnicalDenseStore(technical_corpus)
        self.sparse_store = TechnicalSparseStore(technical_corpus)
        self.generator = generator

    def extract_search_query(self, query: str) -> str:
        """Uses the first sentence of a long ticket as the retrieval query (splits on '.' and newlines only)."""
        sentences = [s.strip() for s in query.replace('\n', '. ').split('.') if s.strip()]
        if sentences:
            return sentences[0]
        return query[:100]

    def reciprocal_rank_fusion(self, dense_res, bm25_res, k=60, top_k=3):
        """Fuses dense and sparse rankings with Reciprocal Rank Fusion (RRF, k=60)."""
        rrf_scores = {}

        def add_ranks(res_docs, res_idxs):
            for rank, idx in enumerate(res_idxs):
                if idx not in rrf_scores:
                    rrf_scores[idx] = {"doc": res_docs[rank], "score": 0.0}
                rrf_scores[idx]["score"] += 1.0 / (k + (rank + 1))

        add_ranks(dense_res[0], dense_res[2])
        add_ranks(bm25_res[0], bm25_res[2])
        sorted_docs = sorted(rrf_scores.values(), key=lambda x: x["score"], reverse=True)
        return [item["doc"] for item in sorted_docs[:top_k]]

    def rerank_documents(self, query: str, retrieved_docs: List[str], top_n: int = 3):
        """Reranks documents with the cross-encoder (the evaluation uses rerank_with_scores, which also returns scores)."""
        if not retrieved_docs:
            return []
        pairs = [[query, doc] for doc in retrieved_docs]
        scores = reranker_model.predict(pairs)
        scored_docs = sorted(zip(retrieved_docs, scores), key=lambda x: x[1], reverse=True)
        return [doc for doc, score in scored_docs[:top_n]]

Loading Cross-Encoder Reranker...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [6]:
from typing import List
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer


class LearnedRouter:
    """Chooses between sparse (BM25) and dense retrieval from the text of a ticket.
        Labels are produced and the router is trained in the router-training cell (Helper 3)."""

    def __init__(self):
        self.vectorizer = TfidfVectorizer(max_features=1000)
        self.classifier = LogisticRegression(random_state=42)
        self._is_trained = False
        self.n_train = 0

    def train(self, sample_queries: List[str], sample_labels: List[str]):
        """Fits TF-IDF features and logistic regression on (ticket text, best-retriever label) pairs."""
        X = self.vectorizer.fit_transform(sample_queries)
        self.classifier.fit(X, sample_labels)
        self._is_trained = True
        self.n_train = len(sample_queries)
        print(f"Router trained on {self.n_train} examples | classes: {list(self.classifier.classes_)}")

    def predict_strategy(self, query: str) -> str:
        """Returns 'sparse_bm25' or 'dense_vector'. Raises if called before training."""
        if not self._is_trained:
            raise RuntimeError("LearnedRouter has not been trained yet.")
        X = self.vectorizer.transform([query])
        return self.classifier.predict(X)[0]

In [7]:
import os
from peft import PeftModel

ADAPTER_PATH = "/content/drive/MyDrive/raft_qwen_adapter"
assert os.path.exists(os.path.join(ADAPTER_PATH, "adapter_model.safetensors")), \
    "Adapter not found. Mount Drive first (cell 2) and check the path."

generator = QwenGenerator(adapter_path=ADAPTER_PATH)
pipeline = EnterpriseAdaptiveRAGPipeline(technical_corpus=enterprise_docs, generator=generator)

# Checks
assert isinstance(generator.model, PeftModel), "Adapter not attached -> this would be a base-model run"
print("Adapter attached:", type(generator.model).__name__)       # PeftModelForCausalLM is correct
print("Corpus size:", len(pipeline.documents))                    # expect 2000
print("Dense embeddings:", tuple(pipeline.dense_store.doc_embeddings.shape))  # expect (2000, 384)

Loading Qwen generator (4-bit)...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Adapter attached: PeftModelForCausalLM
Corpus size: 2000
Dense embeddings: (2000, 384)


In [4]:
import json, os, time
import numpy as np
import pandas as pd
from rouge_score import rouge_scorer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)

def rerank_with_scores(query, docs, top_n=3):
    scores = reranker_model.predict([[query, d] for d in docs])
    order = np.argsort(scores)[::-1][:top_n]
    return [docs[i] for i in order], [float(scores[i]) for i in order]

def retrieve_all(pipe, ticket, k=3):
    """One call = every retrieval variant + the scores the gate may use + stage timings."""
    sq = pipe.extract_search_query(ticket)
    T = {}
    t0 = time.time(); sp_docs, sp_sc, sp_idx = pipe.sparse_store.search_scored(sq, top_k=5); T["sparse"] = time.time() - t0
    t0 = time.time(); dn_docs, dn_sc, dn_idx = pipe.dense_store.search_scored(sq, top_k=5);  T["dense"] = time.time() - t0
    t0 = time.time()
    fused = pipe.reciprocal_rank_fusion((dn_docs, dn_sc, dn_idx), (sp_docs, sp_sc, sp_idx), top_k=5)
    T["fuse"] = time.time() - t0
    t0 = time.time(); rr_docs, rr_sc = rerank_with_scores(sq, fused, top_n=k); T["rerank"] = time.time() - t0
    return {"sparse": sp_docs[:k], "dense": dn_docs[:k], "hybrid": fused[:k], "hybrid_rerank": rr_docs,
            "gate": {"sparse": float(sp_sc[0]), "dense": float(dn_sc[0]), "ce": float(rr_sc[0])}, "time": T}

In [5]:
N_CAL, N_ROUTER, N_TEST_EXTRA = 100, 700, 250
a = 2100; b = a + N_CAL; c = b + N_ROUTER; d = c + N_TEST_EXTRA
print("Tickets available:", len(t), "| needed:", d)
assert len(t) >= d, ("Not enough tickets. Lower N_ROUTER / N_TEST_EXTRA, or drop the IT-queue filter in the "
                     "dataset cell (this changes the corpus, so rebuild the pipeline afterwards).")
assert enterprise_docs == t["answer"].iloc[100:2100].tolist(), "Corpus is not rows 100:2100 -> leakage risk"

cal_df    = t.iloc[a:b].reset_index(drop=True)
router_df = t.iloc[b:c].reset_index(drop=True)
test_df   = pd.concat([t.iloc[0:100], t.iloc[c:d]]).drop_duplicates(subset="body").reset_index(drop=True)
print(f"calibration={len(cal_df)} | router-train={len(router_df)} | TEST={len(test_df)} distinct tickets")
assert test_df["body"].nunique() == len(test_df)
assert not set(test_df["body"]) & set(router_df["body"]), "test/router overlap"
assert not set(test_df["body"]) & set(cal_df["body"]), "test/calibration overlap"

Tickets available: 23537 | needed: 3150
calibration=100 | router-train=700 | TEST=350 distinct tickets


In [10]:
def router_labels(df):
    """Label each ticket with the retriever whose top-1 doc has higher ROUGE-L vs the reference.
    Ties (often the same document retrieved by both) carry no signal and are skipped."""
    labels, keep = [], []
    for k, (ticket, ref) in enumerate(zip(df["body"], df["answer"])):
        sq = pipeline.extract_search_query(ticket)
        sp = pipeline.sparse_store.search_scored(sq, top_k=1)[0][0]
        dn = pipeline.dense_store.search_scored(sq, top_k=1)[0][0]
        s_sp = scorer.score(ref, sp)["rougeL"].fmeasure
        s_dn = scorer.score(ref, dn)["rougeL"].fmeasure
        if abs(s_sp - s_dn) < 1e-9:
            continue
        labels.append("sparse_bm25" if s_sp > s_dn else "dense_vector")
        keep.append(k)
    return labels, keep

y, keep = router_labels(router_df)
X = [router_df["body"].iloc[k][:500] for k in keep]
print(f"Kept {len(keep)} of {len(router_df)} router tickets (ties dropped)")
print("Label distribution:", pd.Series(y).value_counts(normalize=True).round(3).to_dict())

Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
r_val = LearnedRouter(); r_val.train(Xtr, ytr)
pred = [r_val.predict_strategy(x) for x in Xva]
maj_label = pd.Series(ytr).mode()[0]
print(f"Held-out router accuracy={accuracy_score(yva, pred):.3f} | macro-F1={f1_score(yva, pred, average='macro'):.3f} "
      f"| majority-class accuracy={(pd.Series(yva) == maj_label).mean():.3f}")

# Final router
learned_router = LearnedRouter(); learned_router.train(X, y)
print("Final router:", learned_router.n_train, "examples | classes:", list(learned_router.classifier.classes_))

Kept 613 of 700 router tickets (ties dropped)
Label distribution: {'dense_vector': 0.545, 'sparse_bm25': 0.455}
Router trained on 490 examples | classes: [np.str_('dense_vector'), np.str_('sparse_bm25')]
Held-out router accuracy=0.553 | macro-F1=0.511 | majority-class accuracy=0.545
Router trained on 613 examples | classes: [np.str_('dense_vector'), np.str_('sparse_bm25')]
Final router: 613 examples | classes: [np.str_('dense_vector'), np.str_('sparse_bm25')]


In [11]:
cal = pd.DataFrame([retrieve_all(pipeline, tkt)["gate"] for tkt in cal_df["body"]])
TAU = {p: {k: float(np.percentile(cal[k], p)) for k in ["sparse", "dense", "ce"]} for p in (10, 25, 50)}
print(json.dumps(TAU, indent=2))

print(cal.describe().round(3))
cal.to_csv("/content/drive/MyDrive/rag_eval/ce_calibration_scores.csv", index=False)

{
  "10": {
    "sparse": 14.350822954739412,
    "dense": 0.5392628073692322,
    "ce": -1.956881546974182
  },
  "25": {
    "sparse": 16.605486662705168,
    "dense": 0.6372080892324448,
    "ce": 0.18063458800315857
  },
  "50": {
    "sparse": 19.453225518055525,
    "dense": 0.7275727987289429,
    "ce": 3.2029147148132324
  }
}
        sparse    dense       ce
count  100.000  100.000  100.000
mean    21.282    0.715    2.708
std      7.332    0.113    3.304
min     10.774    0.438   -6.205
25%     16.605    0.637    0.181
50%     19.453    0.728    3.203
75%     24.097    0.786    5.044
max     61.905    0.961    8.773


In [14]:
import os, shutil, time

D = "/content/drive/MyDrive/rag_eval"
OUT = os.path.join(D, "results_v2_raft.jsonl")

def archive_if_exists(path):
    """Move `path` to a uniquely named archive file. Does nothing if it doesn't exist."""
    if not os.path.exists(path):
        print("nothing to archive (file not found):", os.path.basename(path))
        return None
    stamp = time.strftime("%Y%m%d_%H%M%S")
    base, ext = os.path.splitext(path)
    new = f"{base}_ARCHIVED_{stamp}{ext}"
    k = 1
    while os.path.exists(new):          # never overwrite an existing archive
        new = f"{base}_ARCHIVED_{stamp}_{k}{ext}"
        k += 1
    os.rename(path, new)
    print("archived ->", os.path.basename(new))
    return new

archive_if_exists(OUT)
print(sorted(os.listdir(D)))

nothing to archive (file not found): results_v2_raft.jsonl
['archive_20261017_095834', 'ce_calibration_scores.csv', 'results_v2_base.cfg.json', 'results_v2_base.jsonl', 'targ_calibration.json']


In [15]:
import os, json, time, hashlib
import numpy as np
import torch
import transformers, peft
from typing import List
from peft import PeftModel

assert os.path.isdir("/content/drive/MyDrive"), "Run cell 2 (Drive mount) first."
for _n in ("retrieve_all", "learned_router", "pipeline", "test_df", "cal_df"):
    assert _n in globals(), f"Missing `{_n}`: run the earlier cells first."
assert list(test_df.index) == list(range(len(test_df))), "test_df index must be 0..n-1 (helper 6 relies on it)."
assert isinstance(pipeline.generator.model, PeftModel), "Expected PeftModel (needed for disable_adapter)."

RAG_DIR   = "/content/drive/MyDrive/rag_eval"
OUT       = f"{RAG_DIR}/results_v2_base.jsonl"
CFG_PATH  = f"{RAG_DIR}/results_v2_base.cfg.json"
CAL_CACHE = f"{RAG_DIR}/targ_calibration.json"
MODES     = ["none", "sparse", "dense", "hybrid", "hybrid_rerank"]
MAX_NEW   = 150
N_RUN     = len(test_df)
os.makedirs(RAG_DIR, exist_ok=True)

def _sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def _h(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8")).hexdigest()[:12]

# Prompts: must match QwenGenerator.generate in cell 4 (enforced by the guard below)
SYS_CTX = ("You are a precise enterprise IT support assistant. Answer the user's ticket using the "
           "provided past support replies. Do not copy them word for word; write a clear, helpful reply.")
SYS_MULTI = " Combine the relevant information from all the replies into one coherent answer."
SYS_NOCTX = ("You are a precise enterprise IT support assistant. No relevant past support replies "
             "are available. Answer briefly from general IT knowledge and say so if you are unsure.")
PROMPT_SHA = _h(SYS_CTX + SYS_MULTI + SYS_NOCTX)

def build_messages(query: str, context_chunks: List[str], multi_hop: bool):
    if context_chunks:
        context = "\n".join(f"[{i+1}] {c[:700]}" for i, c in enumerate(context_chunks))
        system = SYS_CTX + (SYS_MULTI if multi_hop else "")
        user = f"Past support replies:\n{context}\n\nTicket: {query[:1500]}"
    else:
        system = SYS_NOCTX
        user = f"Ticket: {query[:1500]}"
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]

#base-model generator (adapter disabled)
class BaseGenerator:
    def __init__(self, model, tokenizer):
        self.model, self.tokenizer = model, tokenizer

    @torch.inference_mode()
    def generate(self, query: str, context_chunks: List[str], multi_hop: bool = False,
                 max_new_tokens: int = MAX_NEW) -> str:
        messages = build_messages(query, context_chunks, multi_hop)
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        with self.model.disable_adapter():
            out = self.model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                      temperature=None, top_p=None, top_k=None,
                                      repetition_penalty=1.1, pad_token_id=self.tokenizer.eos_token_id)
        return self.tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

#TARG gate (base-model raw logits)
class TARGMarginGate:
    SYSTEM = "You are an enterprise IT support agent. Answer the ticket concisely."
    def __init__(self, model, tokenizer, max_draft_tokens=25, skip_prefix_tokens=8):
        self.model, self.tokenizer = model, tokenizer
        self.max_draft_tokens, self.skip_prefix_tokens = max_draft_tokens, skip_prefix_tokens

    @torch.inference_mode()
    def evaluate_query(self, query: str) -> dict:
        messages = [{"role": "system", "content": self.SYSTEM},
                    {"role": "user", "content": f"Ticket: {query[:1500]}"}]
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        _sync(); t0 = time.perf_counter()
        with self.model.disable_adapter():
            out = self.model.generate(**inputs, max_new_tokens=self.max_draft_tokens, do_sample=False,
                                      temperature=None, top_p=None, top_k=None,
                                      repetition_penalty=1.0,
                                      output_logits=True, return_dict_in_generate=True,
                                      pad_token_id=self.tokenizer.eos_token_id)
        _sync(); draft_time = time.perf_counter() - t0
        draft = self.tokenizer.decode(out.sequences[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        logits = out.logits
        if len(logits) <= self.skip_prefix_tokens:
            return {"mean_margin": None, "short_draft": True, "n_tokens": len(logits),
                    "draft": draft, "draft_time": draft_time}
        margins = []
        for step in logits[self.skip_prefix_tokens:]:
            top2 = torch.topk(step[0].float(), k=2).values
            margins.append((top2[0] - top2[1]).item())
        return {"mean_margin": float(np.mean(margins)), "short_draft": False, "n_tokens": len(logits),
                "draft": draft, "draft_time": draft_time}

base_gen  = BaseGenerator(pipeline.generator.model, pipeline.generator.tokenizer)
targ_gate = TARGMarginGate(pipeline.generator.model, pipeline.generator.tokenizer)

#guard: base prompts must equal the pipeline generator's prompts
def _captured_messages(call):
    tok = pipeline.generator.tokenizer
    store, orig = [], tok.apply_chat_template
    def spy(messages, **kw):
        store.append(messages)
        return orig(messages, **kw)
    tok.apply_chat_template = spy
    try:
        call()
    finally:
        try:
            del tok.apply_chat_template
        except AttributeError:
            tok.apply_chat_template = orig
    return store[0]

_q = test_df.loc[0, "body"]
for _ctx, _mh in ((["ctx A", "ctx B"], True), (["ctx A"], False), ([], False)):
    _a = _captured_messages(lambda: pipeline.generator.generate(_q, _ctx, multi_hop=_mh, max_new_tokens=1))
    _b = build_messages(_q, _ctx, _mh)
    assert _a == _b, f"Prompt mismatch with QwenGenerator (ctx={len(_ctx)}, multi_hop={_mh}). Fix SYS_* constants."
print("Prompt check passed: base prompts == pipeline generator prompts | prompt_sha =", PROMPT_SHA)

#settings file
cfg = {"model": "Qwen/Qwen2.5-1.5B-Instruct", "adapter": None, "generation": "base model, adapter disabled",
       "max_new_tokens": MAX_NEW, "repetition_penalty": 1.1, "ctx_chars": 700, "ticket_chars": 1500,
       "prompt_sha": PROMPT_SHA,
       "targ": {"draft_tokens": 25, "skip_prefix": 8, "system": TARGMarginGate.SYSTEM},
       "modes": MODES, "transformers": transformers.__version__, "peft": peft.__version__}
if os.path.exists(CFG_PATH):
    old = json.load(open(CFG_PATH))
    if old != cfg:
        for k in sorted(set(old) | set(cfg)):
            if old.get(k) != cfg.get(k):
                print(f"  differs: {k} | on disk: {old.get(k)} | now: {cfg.get(k)}")
        raise AssertionError("Settings changed. Archive the old results + cfg + calibration cache and rerun.")
else:
    assert not (os.path.exists(OUT) and os.path.getsize(OUT) > 0), \
        f"{OUT} exists without a cfg file (old/unknown settings). Archive it first."
    json.dump(cfg, open(CFG_PATH, "w"), indent=2)

#warm-up (not recorded)
_w = test_df.loc[0, "body"]
for _ in range(2):
    targ_gate.evaluate_query(_w)
    base_gen.generate(_w, ["warm-up context"], multi_hop=False, max_new_tokens=20)

#TARG calibration on cal_df (cached)
def targ_calibrate(rates=(50, 75, 90)):
    sig = _h("|".join(cal_df["body"]) + json.dumps(cfg["targ"]) + cfg["model"]
             + cfg["transformers"] + cfg["peft"])
    if os.path.exists(CAL_CACHE):
        c = json.load(open(CAL_CACHE))
        if c.get("sig") == sig:
            c["TAU_M"] = {int(k): v for k, v in c["TAU_M"].items()}
            return c
    margins = [targ_gate.evaluate_query(t)["mean_margin"] for t in cal_df["body"]]
    m = np.array([x for x in margins if x is not None], dtype=float)
    c = {"sig": sig, "margins": margins, "n_short": int(sum(x is None for x in margins)),
         "TAU_M": {r: float(np.percentile(m, r)) for r in rates}}
    json.dump(c, open(CAL_CACHE, "w"))
    return c

_cal = targ_calibrate()
TAU_M = _cal["TAU_M"]
print(f"TARG calibration: n={len(_cal['margins'])}, short drafts={_cal['n_short']}, TAU_M={TAU_M}")

#resume: tolerate a half-written last line only
def load_done(path):
    done = {}
    if not os.path.exists(path):
        return done
    with open(path, "rb") as fh:
        lines = fh.readlines()
    good = 0
    for k, raw in enumerate(lines):
        try:
            rec = json.loads(raw)
        except json.JSONDecodeError:
            assert k == len(lines) - 1, f"Corrupt line {k} in the middle of {path}; inspect manually."
            break
        assert "h" in rec, "Row without ticket hash (old schema). Archive the file."
        done[rec["i"]] = rec["h"]
        good += len(raw)
    with open(path, "r+b") as fh:
        fh.truncate(good)
    return done

done = load_done(OUT)
print("Already completed tickets in checkpoint:", len(done))

#evaluation loop
N_TOTAL = min(N_RUN, len(test_df))
t_start, n_new = time.time(), 0
with open(OUT, "a") as f:
    for i in range(N_TOTAL):
        ticket = test_df.loc[i, "body"]; h = _h(ticket)
        if i in done:
            assert done[i] == h, f"Ticket {i} changed since it was written (test_df differs). Start a fresh file."
            continue
        targ = targ_gate.evaluate_query(ticket)
        r = retrieve_all(pipeline, ticket)
        rec = {"i": i, "h": h, "gate": r["gate"],
               "router": str(learned_router.predict_strategy(ticket[:500])),
               "time": r["time"],
               "mean_margin": targ["mean_margin"], "short_draft": targ["short_draft"],
               "n_draft_tokens": targ["n_tokens"], "draft": targ["draft"],
               "ans": {}, "lat": {"targ_draft": targ["draft_time"]}}
        for m in MODES:
            ctx = [] if m == "none" else r[m]
            _sync(); t0 = time.perf_counter()
            rec["ans"][m] = base_gen.generate(ticket, ctx, multi_hop=len(ctx) > 1, max_new_tokens=MAX_NEW)
            _sync(); rec["lat"][m] = time.perf_counter() - t0
        f.write(json.dumps(rec, default=float) + "\n"); f.flush(); os.fsync(f.fileno())
        n_new += 1
        if n_new % 10 == 0 or i == N_TOTAL - 1:
            el = time.time() - t_start
            print(f"  ticket {i+1}/{N_TOTAL} | {el/n_new:.1f}s/ticket | ETA {(N_TOTAL-i-1)*el/n_new/60:.0f} min")
print("Done.")

Prompt check passed: base prompts == pipeline generator prompts | prompt_sha = d2ff2faa8c9c
TARG calibration: n=100, short drafts=0, TAU_M={50: 2.7869944852941178, 75: 3.6704963235294117, 90: 4.827757352941177}
Already completed tickets in checkpoint: 7
  ticket 17/350 | 31.8s/ticket | ETA 177 min
  ticket 27/350 | 30.9s/ticket | ETA 167 min
  ticket 37/350 | 30.9s/ticket | ETA 161 min
  ticket 47/350 | 30.8s/ticket | ETA 155 min
  ticket 57/350 | 31.2s/ticket | ETA 153 min
  ticket 67/350 | 31.1s/ticket | ETA 147 min
  ticket 77/350 | 31.0s/ticket | ETA 141 min
  ticket 87/350 | 30.9s/ticket | ETA 135 min
  ticket 97/350 | 31.0s/ticket | ETA 131 min
  ticket 107/350 | 31.1s/ticket | ETA 126 min
  ticket 117/350 | 31.2s/ticket | ETA 121 min
  ticket 127/350 | 31.0s/ticket | ETA 115 min
  ticket 137/350 | 31.1s/ticket | ETA 110 min
  ticket 147/350 | 31.0s/ticket | ETA 105 min
  ticket 157/350 | 31.0s/ticket | ETA 100 min
  ticket 167/350 | 31.1s/ticket | ETA 95 min
  ticket 177/350 | 3

In [16]:
import json, os, hashlib, platform, sys, datetime
import numpy as np, torch, transformers, peft

RAG_DIR = "/content/drive/MyDrive/rag_eval"
OUT = f"{RAG_DIR}/results_v2_base.jsonl"
h = lambda s: hashlib.sha1(s.encode("utf-8")).hexdigest()[:12]

#integrity of the results file
rows = [json.loads(l) for l in open(OUT) if l.strip()]
ids = [r["i"] for r in rows]
assert len(ids) == len(set(ids)) == len(test_df), (len(ids), len(set(ids)), len(test_df))
assert sorted(ids) == list(range(len(test_df)))
assert all(r["h"] == h(test_df.loc[r["i"], "body"]) for r in rows), "hash mismatch"
print("results OK:", len(rows), "unique rows, all hashes match test_df")
print("short drafts:", sum(r["short_draft"] for r in rows),
      "| empty answers:", sum(not a.strip() for r in rows for a in r["ans"].values()))
print("draft tokens: min %d, median %d" % (min(r["n_draft_tokens"] for r in rows),
                                           np.median([r["n_draft_tokens"] for r in rows])))

#environment record (the session that ran this cell)
try:
    import bitsandbytes as bnb; bnb_v = bnb.__version__
except Exception:
    bnb_v = None
env = {"saved_at": datetime.datetime.now().isoformat(),
       "python": sys.version.split()[0], "platform": platform.platform(),
       "torch": torch.__version__, "cuda": torch.version.cuda,
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
       "gpu_mem_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if torch.cuda.is_available() else None,
       "transformers": transformers.__version__, "peft": peft.__version__, "bitsandbytes": bnb_v,
       "results_sha256": hashlib.sha256(open(OUT, "rb").read()).hexdigest(),
       "n_rows": len(rows)}
json.dump(env, open(f"{RAG_DIR}/environment.json", "w"), indent=2)
print(json.dumps(env, indent=2))

results OK: 350 unique rows, all hashes match test_df
short drafts: 0 | empty answers: 0
draft tokens: min 11, median 25
{
  "saved_at": "2026-10-17T13:16:12.031591",
  "python": "3.13.15",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "torch": "2.11.0+cu130",
  "cuda": "13.0",
  "gpu": "NVIDIA L4",
  "gpu_mem_gb": 23.7,
  "transformers": "5.18.0",
  "peft": "0.21.1",
  "bitsandbytes": "0.50.2",
  "results_sha256": "75c21816ebd21f2e8680327394710ef9a8f3694f3e1cc5bafa039983e57be078",
  "n_rows": 350
}


In [30]:
gain = RL["hybrid_rerank"] - RL["none"]

print(
    f"\nMean gain from retrieval (hybrid_rerank - none): {gain.mean():.4f}; "
    f"tickets where retrieval helps: {(gain > 0).mean():.3f}"
)

g_rows = []

for gname, need in (
    ("CE score", ce),
    ("TARG margin", -margin)
):
    ok = np.isfinite(need) & np.isfinite(gain)

    if ok.sum() < 2:
        rho, prho, auc = np.nan, np.nan, np.nan
    else:
        rho, prho = spearmanr(need[ok], gain[ok])
        y = (gain[ok] > 0).astype(int)

        auc = (
            roc_auc_score(y, need[ok])
            if 0 < y.sum() < len(y)
            else np.nan
        )

    g_rows.append({
        "gate score": gname,
        "n": int(ok.sum()),
        "Spearman(need, gain)": round(float(rho), 3),
        "p": round(float(prho), 4),
        "AUROC(gain>0)": round(float(auc), 3)
    })

table3 = pd.DataFrame(g_rows)

print(
    "\nTABLE 3 (gate quality; ~0.5 AUROC / ~0 rho = weak signal)\n",
    table3.to_string(index=False)
)

for name, retrieved in RETR.items():
    if 0 < retrieved.sum() < len(retrieved):
        skipped = ~retrieved
        print(
            f"{name}: mean ROUGE-L retrieved "
            f"{RL['hybrid_rerank'][retrieved].mean():.4f} "
            f"(n={retrieved.sum()}), skipped "
            f"{RL['none'][skipped].mean():.4f} "
            f"(n={skipped.sum()})"
        )

_tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

def ntok(answer):
    return len(_tok(answer).input_ids)

len_rows = [
    {
        "config": name,
        "mean answer tokens": round(
            float(np.mean([ntok(a) for a in ANS[name]])), 1
        ),
        "% at 150-token cap": round(
            100 * float(np.mean([ntok(a) >= 150 for a in ANS[name]])), 1
        )
    }
    for name in MODES
]

print(
    "\nANSWER LENGTH (reference mean tokens: %.1f)"
    % np.mean([ntok(ref.loc[i]) for i in idx])
)
print(pd.DataFrame(len_rows).to_string(index=False))

SAVE_TAG = os.path.basename(OUT).replace("results_v2_", "").replace(".jsonl", "")

for name, table in (
    ("table1", table1),
    ("table2", table2),
    ("table3", table3)
):
    table.to_csv(f"{SAVE}/{name}_{SAVE_TAG}.csv", index=False)

len_df = pd.DataFrame(len_rows)
len_df.to_csv(f"{SAVE}/answer_lengths_{SAVE_TAG}.csv", index=False)

print(
    "Saved tagged tables:",
    sorted(
        filename for filename in os.listdir(SAVE)
        if filename.endswith(f"_{SAVE_TAG}.csv")
    )
)



Mean gain from retrieval (hybrid_rerank - none): 0.0340; tickets where retrieval helps: 0.694

TABLE 3 (gate quality; ~0.5 AUROC / ~0 rho = weak signal)
  gate score   n  Spearman(need, gain)      p  AUROC(gain>0)
   CE score 350                 0.179 0.0008          0.583
TARG margin 350                 0.102 0.0556          0.576
ce_gate_p10: mean ROUGE-L retrieved 0.2105 (n=322), skipped 0.1931 (n=28)
ce_gate_p25: mean ROUGE-L retrieved 0.2136 (n=297), skipped 0.1736 (n=53)
ce_gate_p50: mean ROUGE-L retrieved 0.2192 (n=189), skipped 0.1770 (n=161)
targ_r50: mean ROUGE-L retrieved 0.2099 (n=165), skipped 0.1824 (n=185)
targ_r75: mean ROUGE-L retrieved 0.2084 (n=248), skipped 0.1801 (n=102)
targ_r90: mean ROUGE-L retrieved 0.2064 (n=317), skipped 0.1852 (n=33)

ANSWER LENGTH (reference mean tokens: 70.6)
       config  mean answer tokens  % at 150-token cap
         none               104.1                32.3
       sparse               128.0                49.1
        dense       

In [2]:

import os
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from statsmodels.stats.multitest import multipletests

assert "RL" in globals() and "R1" in globals(), \
    "Run Cell 12 first."
assert "RETR" in globals() and "FAMILY" in globals(), \
    "Missing gate results or predefined comparison family."
assert "SAVE" in globals() and "N" in globals(), \
    "Missing SAVE or N."

rng2 = np.random.default_rng(123)

hr = np.asarray(RL["hybrid_rerank"], dtype=float)
no = np.asarray(RL["none"], dtype=float)
gain = hr - no

assert len(hr) == len(no) == N, \
    "ROUGE arrays do not match N."
assert N > 0, "No test observations."

GATES = [
    "ce_gate_p10", "ce_gate_p25", "ce_gate_p50",
    "targ_r50", "targ_r75", "targ_r90"
]
for name in GATES:
    assert name in RETR, f"Missing retrieval decisions: {name}"
    assert name in RL, f"Missing ROUGE scores: {name}"

def boot_mean(x, B=2000, rng=None):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]

    if len(x) == 0:
        return np.nan, np.nan, np.nan

    if rng is None:
        rng = rng2

    b = x[rng.integers(0, len(x), size=(B, len(x)))].mean(axis=1)
    return (
        float(x.mean()),
        float(np.percentile(b, 2.5)),
        float(np.percentile(b, 97.5))
    )


def paired_wilcoxon(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    valid = np.isfinite(x) & np.isfinite(y)
    x, y = x[valid], y[valid]

    if len(x) == 0:
        return np.nan

    d = x - y
    if np.all(d == 0):
        return 1.0

    return float(stats.wilcoxon(x, y).pvalue)

#Gate versus a random gate at the SAME realized rate

def excess_vec(name, mask):
    mask = np.asarray(mask, dtype=bool)
    sel = np.asarray(RETR[name], dtype=bool)[mask]
    h = hr[mask]
    n = no[mask]

    if len(sel) == 0:
        return np.array([], dtype=float)

    q = sel.mean()
    gate = np.where(sel, h, n)
    random_expected = q * h + (1.0 - q) * n

    # Ticket-level difference from the rate-matched random expectation.
    return gate - random_expected


def gate_vs_random(name, mask=None, S=10000):
    if mask is None:
        mask = np.ones(N, dtype=bool)

    mask = np.asarray(mask, dtype=bool)
    sel = np.asarray(RETR[name], dtype=bool)[mask]
    h = hr[mask]
    n = no[mask]

    if len(sel) == 0:
        raise ValueError(f"Empty sample for gate {name}.")

    k = int(sel.sum())
    q = float(sel.mean())
    gate = np.where(sel, h, n)

    excess = excess_vec(name, mask)
    m, lo, hi = boot_mean(excess, rng=rng2)

    # Conditional randomization test:
    # retain exactly k retrieved tickets in each random draw.
    d = h - n
    base = n.sum()
    sims = np.empty(S, dtype=float)

    for s in range(S):
        chosen = rng2.permutation(len(d))[:k]
        sims[s] = (base + d[chosen].sum()) / len(d)

    observed = float(gate.mean())
    p_ge = (1 + np.count_nonzero(sims >= observed)) / (S + 1)
    p_le = (1 + np.count_nonzero(sims <= observed)) / (S + 1)
    p_two = min(1.0, 2.0 * min(p_ge, p_le))

    return {
        "gate": name,
        "realized rate": q,
        "ROUGE-L": observed,
        "excess over random": m,
        "95% CI": f"[{lo:.4f}, {hi:.4f}]",
        "perm p (two-sided)": float(p_two),
        "_p2": float(p_two)
    }


# Gate comparisons against rate-matched random baselines.
gv = pd.DataFrame([
    gate_vs_random(g, np.ones(N, dtype=bool))
    for g in GATES
])

print(f"GATES VS RATE-MATCHED RANDOM GATE (n = {N})")
print(gv.drop(columns="_p2").round(4).to_string(index=False))


# Main descriptive contrast: TARG r75 versus CE p25,
# adjusting each gate for its own realized random baseline.
d_ex = (
    excess_vec("targ_r75", np.ones(N, dtype=bool))
    - excess_vec("ce_gate_p25", np.ones(N, dtype=bool))
)

m, lo, hi = boot_mean(d_ex, rng=rng2)

print(
    "\nRate-adjusted TARG r75 minus CE p25 "
    "(difference in excess over each gate's own random baseline): "
    f"{m:.4f} [{lo:.4f}, {hi:.4f}]"
)

#Corrected Table 2: the original, predefined 12 comparisons

assert len(FAMILY) == 12, \
    f"Expected the predefined family of 12 comparisons; got {len(FAMILY)}."

rows = []
pvals = []

for A, B in FAMILY:

    if A not in RL:
        raise KeyError(f"Missing ROUGE configuration: {A}")

    if B == "random_r75":
        # Compare each gate against a random policy matched to
        # that gate's ACTUAL test retrieval rate.
        result = gate_vs_random(A, np.ones(N, dtype=bool))

        rows.append({
            "comparison": f"{A} vs rate-matched random",
            "mean diff": round(result["excess over random"], 4),
            "95% CI": result["95% CI"],
            "n nonzero diffs": np.nan,
            "test": "permutation"
        })
        pvals.append(result["_p2"])

    else:
        if B not in RL:
            raise KeyError(f"Missing ROUGE configuration: {B}")

        x = np.asarray(RL[A], dtype=float)
        yb = np.asarray(RL[B], dtype=float)

        valid = np.isfinite(x) & np.isfinite(yb)
        d = x[valid] - yb[valid]

        mean_diff, lo, hi = boot_mean(d, rng=rng2)
        p = paired_wilcoxon(x, yb)

        rows.append({
            "comparison": f"{A} vs {B}",
            "mean diff": round(mean_diff, 4),
            "95% CI": f"[{lo:.4f}, {hi:.4f}]",
            "n nonzero diffs": int(np.count_nonzero(d)),
            "test": "Wilcoxon"
        })
        pvals.append(p)

# Holm correction across the entire predefined family of 12.
reject, p_holm, _, _ = multipletests(
    pvals, alpha=0.05, method="holm"
)

for row, p, ph, rej in zip(rows, pvals, p_holm, reject):
    row["raw p"] = round(float(p), 6)
    row["Holm p"] = round(float(ph), 6)
    row["significant"] = bool(rej)

table2_corr = pd.DataFrame(rows)

print(f"\nCORRECTED TABLE 2 (Holm over m = {len(FAMILY)})")
print(table2_corr.to_string(index=False))

#Corrected Table 3: gate-quality analysis
# Exploratory analysis, outside the predefined Holm family.

def auroc_bootstrap(score, target, B=2000):
    score = np.asarray(score, dtype=float)
    target = np.asarray(target, dtype=int)

    valid = np.isfinite(score) & np.isfinite(target)
    score = score[valid]
    target = target[valid]

    if len(target) == 0 or len(np.unique(target)) < 2:
        return np.nan, np.nan, np.nan, len(target)

    auc = float(roc_auc_score(target, score))
    boot = []

    for _ in range(B):
        j = rng2.integers(0, len(target), len(target))
        if len(np.unique(target[j])) < 2:
            continue
        boot.append(roc_auc_score(target[j], score[j]))

    if not boot:
        return auc, np.nan, np.nan, len(target)

    return (
        auc,
        float(np.percentile(boot, 2.5)),
        float(np.percentile(boot, 97.5)),
        len(target)
    )


y_gain = (gain > 0).astype(int)

gate_scores = [
    ("CE score (retrieve if high)", np.asarray(ce, dtype=float)),
    ("TARG margin (retrieve if low)", -np.asarray(margin, dtype=float))
]

g_rows = []

for name, score in gate_scores:
    auc, lo, hi, n_valid = auroc_bootstrap(score, y_gain)

    valid = np.isfinite(score) & np.isfinite(gain)

    if valid.sum() >= 2:
        rho, p_rho = spearmanr(score[valid], gain[valid])
    else:
        rho, p_rho = np.nan, np.nan

    g_rows.append({
        "gate score": name,
        "n": int(n_valid),
        "AUROC(gain>0)": round(auc, 3) if np.isfinite(auc) else np.nan,
        "95% CI": (
            f"[{lo:.3f}, {hi:.3f}]"
            if np.isfinite(lo) and np.isfinite(hi)
            else "unavailable"
        ),
        "Spearman(score, gain)": (
            round(float(rho), 3) if np.isfinite(rho) else np.nan
        ),
        "p": round(float(p_rho), 6) if np.isfinite(p_rho) else np.nan
    })

table3_corr = pd.DataFrame(g_rows)

print("\nCORRECTED TABLE 3 (exploratory; outside Holm family)")
print(table3_corr.to_string(index=False))

# 5. Oracle ceiling: descriptive upper bound, not a deployable gate

print(
    f"\nORACLE: always none {no.mean():.4f} | "
    f"always hybrid_rerank {hr.mean():.4f} | "
    f"oracle gate {np.maximum(hr, no).mean():.4f} | "
    f"retrieval hurts on {(gain < 0).mean():.3f} | "
    f"ties {(gain == 0).mean():.3f} | "
    f"helps {(gain > 0).mean():.3f}"
)

#Save corrected tables only.

os.makedirs(SAVE, exist_ok=True)

gv.drop(columns="_p2").to_csv(
    f"{SAVE}/gates_vs_random_base.csv", index=False
)
table2_corr.to_csv(
    f"{SAVE}/table2_corrected_base.csv", index=False
)
table3_corr.to_csv(
    f"{SAVE}/table3_corrected_base.csv", index=False
)

print("\nSaved corrected statistical tables to:", SAVE)


AssertionError: Run Cell 12 first.

In [32]:
import os
import numpy as np, pandas as pd
from scipy import stats
from sentence_transformers import SentenceTransformer
from statsmodels.stats.multitest import multipletests

# ---- 1. recompute leakage flags (same method as cell 15) and save them ----
assert "enterprise_docs" in globals(), "corpus variable `enterprise_docs` not found"
texts = [str(t) for t in enterprise_docs]
assert len(texts) == 2000, len(texts)
emb = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
C = emb.encode(texts, normalize_embeddings=True, batch_size=64)
A = emb.encode(list(test_df["answer"]), normalize_embeddings=True, batch_size=64)
S = A @ C.T
mc, best = S.max(axis=1), S.argmax(axis=1)
rl_best = np.array([scorer.score(test_df["answer"].iloc[k], texts[best[k]])["rougeL"].fmeasure
                    for k in range(len(test_df))])
n90, n95 = int((mc > 0.90).sum()), int((mc > 0.95).sum())
print(f"leakage flags recomputed: >0.90: {n90} (earlier run: 54) | >0.95: {n95} (earlier run: 28)")
assert abs(n90 - 54) <= 2 and abs(n95 - 28) <= 2, "does not reproduce the earlier leakage check; stop and investigate"
pd.DataFrame({"i": test_df.index, "max_cos_ref_vs_corpus": mc, "rougeL_nearest_doc": rl_best}) \
  .to_csv(f"{SAVE}/leakage_flags.csv", index=False)

# ---- 2. save the corrected tables from 12b ----
gv.drop(columns="_p2").to_csv(f"{SAVE}/gates_vs_random_base.csv", index=False)
table2_corr.to_csv(f"{SAVE}/table2_corrected_base.csv", index=False)
table3_corr.to_csv(f"{SAVE}/table3_corrected_base.csv", index=False)

# ---- 3. PRE-DECLARED leakage sensitivity (cutoffs 0.90 and 0.95; secondary, outside the main Holm family) ----
mc_ = np.asarray(mc)
assert len(mc_) == N, (len(mc_), N)
SENS_FAM = [(a, b) for a, b in FAMILY if b != "random_r75"]     # Holm within this subset (m = 9)

def sensitivity(cut):
    mask = mc_ <= cut
    print(f"\n=== SENSITIVITY: tickets with reference-corpus cosine <= {cut:.2f}: n = {mask.sum()} of {N} ===")
    t1 = []
    for n_ in ["none", "sparse", "dense", "hybrid", "hybrid_rerank", "router"] + GATES:
        m_, lo, hi = boot_mean(RL[n_][mask])
        t1.append({"config": n_, "ROUGE-L": round(m_, 4), "95% CI": f"[{lo:.4f}, {hi:.4f}]"})
    print(pd.DataFrame(t1).to_string(index=False))
    rows, pv = [], []
    for A_, B_ in SENS_FAM:
        try:
            p = stats.wilcoxon(RL[A_][mask], RL[B_][mask]).pvalue
        except ValueError:
            p = 1.0
        m_, lo, hi = boot_mean(RL[A_][mask] - RL[B_][mask])
        rows.append({"comparison": f"{A_} vs {B_}", "mean diff": round(m_, 4), "95% CI": f"[{lo:.4f}, {hi:.4f}]"})
        pv.append(p)
    rj, ph, _, _ = multipletests(pv, alpha=0.05, method="holm")
    for r_, p_, h_, j_ in zip(rows, pv, ph, rj):
        r_.update({"raw p": round(p_, 4), "Holm p (m=%d)" % len(SENS_FAM): round(h_, 4), "significant": bool(j_)})
    print(pd.DataFrame(rows).to_string(index=False))
    if mask.sum() < 2:
      raise ValueError(f"Too few eligible tickets for cutoff {cut:.2f}")

    gv_ = pd.DataFrame([gate_vs_random(g, mask) for g in GATES]).drop(columns="_p2")
    print("\nGates vs rate-matched random (subset):\n", gv_.to_string(index=False))
    return pd.DataFrame(t1), pd.DataFrame(rows), gv_

for cut in (0.90, 0.95):
    a, b, c = sensitivity(cut)
    a.to_csv(f"{SAVE}/sens_{int(cut*100)}_table1_base.csv", index=False)
    b.to_csv(f"{SAVE}/sens_{int(cut*100)}_table2_base.csv", index=False)
    c.to_csv(f"{SAVE}/sens_{int(cut*100)}_gates_base.csv", index=False)
print("\nsaved leakage flags, corrected tables and sensitivity tables to", SAVE)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

leakage flags recomputed: >0.90: 54 (earlier run: 54) | >0.95: 28 (earlier run: 28)

=== SENSITIVITY: tickets with reference-corpus cosine <= 0.90: n = 296 of 350 ===
       config  ROUGE-L           95% CI
         none   0.1773 [0.1708, 0.1844]
       sparse   0.2030 [0.1950, 0.2114]
        dense   0.2074 [0.1995, 0.2156]
       hybrid   0.2065 [0.1998, 0.2137]
hybrid_rerank   0.2051 [0.1975, 0.2127]
       router   0.2054 [0.1975, 0.2137]
  ce_gate_p10   0.2048 [0.1970, 0.2124]
  ce_gate_p25   0.2030 [0.1950, 0.2108]
  ce_gate_p50   0.1949 [0.1877, 0.2020]
     targ_r50   0.1947 [0.1873, 0.2019]
     targ_r75   0.1980 [0.1904, 0.2055]
     targ_r90   0.2015 [0.1944, 0.2091]
                  comparison  mean diff             95% CI  raw p  Holm p (m=10)  significant
              sparse vs none     0.0257   [0.0188, 0.0332] 0.0000         0.0000         True
               dense vs none     0.0301   [0.0223, 0.0377] 0.0000         0.0000         True
             hybrid vs dense   

In [25]:
##SANITY CHECK


import json, re, hashlib
import numpy as np

CHECK_FILE = "/content/drive/MyDrive/rag_eval/results_v2_base.jsonl"   # change to ..._raft.jsonl for the RAFT run
recs = [json.loads(l) for l in open(CHECK_FILE) if l.strip()]
ref = test_df["answer"]
_h = lambda s: hashlib.sha1(s.encode("utf-8")).hexdigest()[:12]

# structural checks
idxs = [r["i"] for r in recs]
print("rows:", len(recs), "of", len(test_df), "| duplicate indices:", len(idxs) - len(set(idxs)))
print("keys:", sorted(recs[0].keys()))
print("lat keys:", sorted(recs[0]["lat"].keys()), "| time keys:", sorted(recs[0]["time"].keys()))
assert all(r["h"] == _h(test_df.loc[r["i"], "body"]) for r in recs), "Ticket hash mismatch with test_df"

margins = np.array([np.nan if r["mean_margin"] is None else r["mean_margin"] for r in recs], dtype=float)
print("short drafts:", int(np.isnan(margins).sum()),
      "| margin min/median/max: %.2f / %.2f / %.2f" % (np.nanmin(margins), np.nanmedian(margins), np.nanmax(margins)))

allans = [(r["i"], m, a) for r in recs for m, a in r["ans"].items()]
print("answers:", len(allans), "| empty:", sum(not a.strip() for _, _, a in allans))

# RAFT-format leakage (meaningful on the RAFT file; expect 0 on the base file)
pat = re.compile(r"chain[- ]of[- ]thought|final answer|##\s*context|context documents", re.I)
print("answers containing RAFT scaffolding:", sum(bool(pat.search(a)) for _, _, a in allans), "of", len(allans))
print("placeholder phrase in any answer:", sum("Please refer to standard support guidelines" in a for _, _, a in allans))
print("answers over 600 chars (rough cap proxy):", sum(len(a) > 600 for _, _, a in allans))

# qualitative samples
for r in recs[:3]:
    mm = "short draft" if r["mean_margin"] is None else "%.2f" % r["mean_margin"]
    print("\n" + "=" * 80, "\nticket", r["i"], "| margin", mm)
    print("TICKET:", test_df.loc[r["i"], "body"][:300].replace("\n", " "))
    print("REF   :", ref[r["i"]][:400].replace("\n", " "))
    print("DRAFT :", r["draft"][:200].replace("\n", " "))
    for m in ("none", "hybrid_rerank"):
        print(f"{m:13s}:", r["ans"][m][:400].replace("\n", " "))

rows: 350 of 350 | duplicate indices: 0
keys: ['ans', 'draft', 'gate', 'h', 'i', 'lat', 'mean_margin', 'n_draft_tokens', 'router', 'short_draft', 'time']
lat keys: ['dense', 'hybrid', 'hybrid_rerank', 'none', 'sparse', 'targ_draft'] | time keys: ['dense', 'fuse', 'rerank', 'sparse']
short drafts: 0 | margin min/median/max: 0.36 / 2.86 / 8.18
answers: 1750 | empty: 0
answers containing RAFT scaffolding: 0 of 1750
placeholder phrase in any answer: 0
answers over 600 chars (rough cap proxy): 1275

ticket 0 | margin 6.26
TICKET: Dear Customer Support, I am reaching out to inquire about the investment analytics tools that are compatible with QuickBooks Online. Could you provide information on the available options and their features? I would appreciate it if you could include detailed information on the tools and their integ
REF   : Thank you for your inquiry regarding investment analytics tools compatible with QuickBooks Online. There are several options available that can integrate seamle

In [26]:
import subprocess, sys, datetime
out = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
path = "/content/drive/MyDrive/rag_eval/pip_freeze.txt"
with open(path, "w") as f:
    f.write(f"# frozen {datetime.datetime.now().isoformat()}\n{out}")
print("saved", path, "|", len(out.splitlines()), "packages")

saved /content/drive/MyDrive/rag_eval/pip_freeze.txt | 725 packages


In [1]:
!nvidia-smi

Sat Oct 17 21:23:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   59C    P8             14W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import transformers
import peft
import torch

print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("PyTorch:", torch.__version__)
print(
    "GPU:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "No GPU"
)

Transformers: 5.18.0
PEFT: 0.21.1
PyTorch: 2.11.0+cu130
GPU: NVIDIA L4
